In [2]:
import pandas as pd
df=pd.read_csv(r'C:\Users\Eshwar\OneDrive\Desktop\VS_Code\Swiggy\swiggy.csv')
display(df)

,id,name,city,rating,rating_count,cost,cuisine,lic_no,link,address,menu
0,567335,AB FOODS POINT,Abohar,--,Too Few Ratings,₹ 200,"Beverages,Pizzas",22122652000138,https://www.swiggy.com/restaurants/ab-foods-po...,"AB FOODS POINT, NEAR RISHI NARANG DENTAL CLINI...",Menu/567335.json
1,531342,Janta Sweet House,Abohar,4.4,50+ ratings,₹ 200,"Sweets,Bakery",12117201000112,https://www.swiggy.com/restaurants/janta-sweet...,"Janta Sweet House, Bazar No.9, Circullar Road,...",Menu/531342.json
2,158203,theka coffee desi,Abohar,3.8,100+ ratings,₹ 100,Beverages,22121652000190,https://www.swiggy.com/restaurants/theka-coffe...,"theka coffee desi, sahtiya sadan road city",Menu/158203.json
3,187912,Singh Hut,Abohar,3.7,20+ ratings,₹ 250,"Fast Food,Indian",22119652000167,https://www.swiggy.com/restaurants/singh-hut-n...,"Singh Hut, CIRCULAR ROAD NEAR NEHRU PARK ABOHAR",Menu/187912.json
4,543530,GRILL MASTERS,Abohar,--,Too Few Ratings,₹ 250,"Italian-American,Fast Food",12122201000053,https://www.swiggy.com/restaurants/grill-maste...,"GRILL MASTERS, ADA Heights, Abohar - Hanumanga...",Menu/543530.json
...,...,...,...,...,...,...,...,...,...,...,...
148536,553122,The Food Delight,Yavatmal,--,Too Few Ratings,₹ 200,"Fast Food,Snacks",21522053000452,https://www.swiggy.com/restaurants/the-food-de...,"The Food Delight, 94MC+X35, New Singhania Naga...",Menu/553122.json
148537,562647,MAITRI FOODS & BEVERAGES,Yavatmal,--,Too Few Ratings,₹ 300,Pizzas,license,https://www.swiggy.com/restaurants/maitri-food...,"MAITRI FOODS & BEVERAGES, POLIC MITRYA SOCIETY...",Menu/562647.json
148538,559435,Cafe Bella Ciao,Yavatmal,--,Too Few Ratings,₹ 300,"Fast Food,Snacks",21522251000378,https://www.swiggy.com/restaurants/cafe-bella-...,"Cafe Bella Ciao, SHOP NO 2 NEMANI MARKET SBI S...",Menu/559435.json
148539,418989,GRILL ZILLA,Yavatmal,--,Too Few Ratings,₹ 250,Continental,21521251000241,https://www.swiggy.com/restaurants/grill-zilla...,"GRILL ZILLA, SHO NO 2/6, POSTEL GROUND CHOWPAT...",Menu/418989.json


In [3]:
df_clean=df.drop_duplicates()
df_clean=df_clean.dropna()


In [4]:
df_clean.duplicated().sum()

np.int64(0)

In [5]:
def splitcity(city):
    if pd.isna(city):
        return pd.Series(['NA','NA'])
    part=city.split(',')
    if len(part)>1:
        return pd.Series([part[0].strip(),part[1].strip()])
    else:
        return pd.Series(['NA',city.strip()])

df_clean[['sub_city','main_city']]=df_clean['city'].apply(splitcity)
df_clean=df_clean.drop(['city'],axis=1)


In [6]:
df_clean['rating_count'] = df_clean['rating_count'].replace({
    'Too Few Ratings': '10',
    '50+ ratings': '51',
    '100+ ratings': '101',
    '1K+ ratings': '1001',
    '5k ratings': '5001',
    '20+ ratings': '21',
    'nan': '0'  # If this is a string "nan", not actual NaN
})

In [7]:
def make_numeric_or_zero(value):
    return value if str(value).isnumeric() else 0

df_clean['lic_no'] = df_clean['lic_no'].apply(make_numeric_or_zero)
df_clean

#name,cuisne,ratint
# Remove rupee symbol and convert to numeric
df_clean['cost'] = df_clean['cost'].replace('[₹,]', '', regex=True).astype(str)
df_clean['cost'] = pd.to_numeric(df_clean['cost'], errors='coerce').fillna(0).astype(int)


In [10]:
import pandas as pd


# Create copy for main_city
df_main = df_clean.copy()
df_main["city"] = df_main["main_city"]

# Create copy for sub_city
df_sub = df_clean.copy()
df_sub["city"] = df_sub["sub_city"]

# Combine both
df_expanded = pd.concat([df_main, df_sub], ignore_index=True)

# --- Keep only required columns ---
df_expanded = df_expanded[['name', 'rating', 'rating_count', 
                           'city', 'cuisine', 'cost', 'address']]

# Save expanded version
df_expanded.to_csv("cleaned_data.csv", index=False)

print(df_expanded.head())


                name rating rating_count    city                     cuisine  \
0     AB FOODS POINT     --           10  Abohar            Beverages,Pizzas   
1  Janta Sweet House    4.4           51  Abohar               Sweets,Bakery   
2  theka coffee desi    3.8          101  Abohar                   Beverages   
3          Singh Hut    3.7           21  Abohar            Fast Food,Indian   
4      GRILL MASTERS     --           10  Abohar  Italian-American,Fast Food   

   cost                                            address  
0   200  AB FOODS POINT, NEAR RISHI NARANG DENTAL CLINI...  
1   200  Janta Sweet House, Bazar No.9, Circullar Road,...  
2   100         theka coffee desi, sahtiya sadan road city  
3   250    Singh Hut, CIRCULAR ROAD NEAR NEHRU PARK ABOHAR  
4   250  GRILL MASTERS, ADA Heights, Abohar - Hanumanga...  


In [ ]:
main city sub city ona append panitu one encoding panum

In [15]:
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

# --------------------------------------------------
# 1 Load cleaned data
# --------------------------------------------------
df = pd.read_csv("cleaned_data.csv").reset_index(drop=True)

# --------------------------------------------------
# 2 Drop name and address (high-cardinality junk columns)
# --------------------------------------------------
drop_cols = ["name", "Name", "address", "Address"]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

# --------------------------------------------------
# 3 Identify categorical columns
# --------------------------------------------------
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print("Categorical columns to label-encode:", categorical_cols)

# Dictionary to store encoders
encoders = {}

# --------------------------------------------------
# 4 Apply Label Encoding to each categorical column
# --------------------------------------------------
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = df[col].astype(str)  # convert to string to avoid errors
    df[col] = le.fit_transform(df[col])
    encoders[col] = le  # save encoder

# --------------------------------------------------
# 5 Save encoded CSV
# --------------------------------------------------
df.to_csv("encoded_data.csv", index=False)

# --------------------------------------------------
# 6 Save all label encoders as PKL
# --------------------------------------------------
joblib.dump(encoders, "label_encoders.pkl")

print("Label encoding completed successfully!")
print("encoded_data.csv and label_encoders.pkl created.")


Categorical columns to label-encode: ['rating', 'rating_count', 'city', 'cuisine']
Label encoding completed successfully!
encoded_data.csv and label_encoders.pkl created.


In [19]:
df = pd.read_csv("encoded_data.csv")
print(df.head())

   rating  rating_count  city  cuisine  cost
0       0             0     1      328   200
1      35             6     1     1983   200
2      29             2     1      289   100
3      28             4     1      796   250
4       0             0     1     1166   250


In [20]:
#index check
import pandas as pd

# Load files
cleaned_df = pd.read_csv("cleaned_data.csv")
encoded_df = pd.read_csv("encoded_data.csv")

# Ensure indices match
cleaned_df = cleaned_df.reset_index(drop=True)
encoded_df = encoded_df.reset_index(drop=True)

# Save back
cleaned_df.to_csv("cleaned_data.csv", index=False)
encoded_df.to_csv("encoded_data.csv", index=False)

# Check
print("Indices match:", cleaned_df.index.equals(encoded_df.index))


Indices match: True


In [12]:
import pandas as pd
import pickle
from sklearn.preprocessing import OneHotEncoder

# Load cleaned_data.csv
df = pd.read_csv("cleaned_data.csv")

# Select categorical columns to encode
cat_cols = ["city", "cuisine"]

# Create OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

# Fit encoder
encoded_array = encoder.fit_transform(df[cat_cols])

# Create encoded dataframe with column names
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(cat_cols))

# Concatenate with original df (excluding original cat columns)
df_final = pd.concat([df.drop(columns=cat_cols), encoded_df], axis=1)

# Save the encoder as pickle
with open("encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

# Save final encoded file
df_final.to_csv("encoded_data.csv", index=False)

print(df_final.head())


                name rating rating_count  cost  \
0     AB FOODS POINT     --           10   200   
1  Janta Sweet House    4.4           51   200   
2  theka coffee desi    3.8          101   100   
3          Singh Hut    3.7           21   250   
4      GRILL MASTERS     --           10   250   

                                             address  city_Abids & Koti  \
0  AB FOODS POINT, NEAR RISHI NARANG DENTAL CLINI...                0.0   
1  Janta Sweet House, Bazar No.9, Circullar Road,...                0.0   
2         theka coffee desi, sahtiya sadan road city                0.0   
3    Singh Hut, CIRCULAR ROAD NEAR NEHRU PARK ABOHAR                0.0   
4  GRILL MASTERS, ADA Heights, Abohar - Hanumanga...                0.0   

   city_Abohar  city_Adajan  city_Adilabad  city_Adityapur  ...  \
0          1.0          0.0            0.0             0.0  ...   
1          1.0          0.0            0.0             0.0  ...   
2          1.0          0.0            0.0     

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import KMeans
import numpy as np
import pickle

# -------------------------------
# 1️⃣ Load the cleaned data
# -------------------------------
df = pd.read_csv("cleaned_data.csv", index_col=0)

# -------------------------------
# 2️⃣ Select columns
# -------------------------------
categorical_cols = ['sub_city', 'main_city', 'cuisine']
numeric_cols = ['rating', 'rating_count', 'cost']

# -------------------------------
# 3️⃣ Clean numeric columns
# -------------------------------
for col in numeric_cols:
    df[col] = (
        df[col]
        .replace(['--', '-', 'NA', 'N/A', 'na', 'NaN', 'null', 'None', ''], np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing numeric values with median
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# -------------------------------
# 4️⃣ Simplify categorical columns (to save memory)
# -------------------------------
for col in categorical_cols:
    top_values = df[col].value_counts().nlargest(20).index  # Keep top 20 unique
    df[col] = df[col].where(df[col].isin(top_values), 'Other')

# -------------------------------
# 5️⃣ Encode categorical columns
# -------------------------------
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_array = enc.fit_transform(df[categorical_cols])

encoded_df = pd.DataFrame(
    encoded_array,s
    columns=enc.get_feature_names_out(categorical_cols),
    index=df.index
)

# -------------------------------
# 6️⃣ Combine encoded + numeric
# -------------------------------
df_encoded = pd.concat([df[numeric_cols], encoded_df], axis=1)

# -------------------------------
# 7️⃣ Apply K-Means Clustering
# -------------------------------
kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42)
df_encoded['Cluster'] = kmeans.fit_predict(df_encoded)

print("✅ K-Means clustering complete.")
print(f"📊 Cluster counts:\n{df_encoded['Cluster'].value_counts()}")

# -------------------------------
# 8️⃣ Map cluster results back to original (non-encoded) DataFrame
# -------------------------------
df_with_clusters = df.copy()
df_with_clusters['Cluster'] = df_encoded['Cluster']

# -------------------------------
# 9️⃣ Save both encoded & mapped DataFrames
# -------------------------------
with open("df_encoded.pkl", "wb") as f:
    pickle.dump(df_encoded, f)

df_with_clusters.to_csv("cleaned_data_with_clusters.csv", index=False)

print("💾 Saved:")
print(" - df_encoded.pkl (for model use)")
print(" - cleaned_data_with_clusters.csv (for human-readable use)")
print(f"📄 Final shape (encoded): {df_encoded.shape}")
print(f"📄 Final shape (mapped): {df_with_clusters.shape}")


✅ K-Means clustering complete.
📊 Cluster counts:
Cluster
2    130521
0     14033
3      2737
4       963
1         1
Name: count, dtype: int64
💾 Saved:
 - df_encoded.pkl (for model use)
 - cleaned_data_with_clusters.csv (for human-readable use)
📄 Final shape (encoded): (148255, 67)
📄 Final shape (mapped): (148255, 7)


In [18]:
from sklearn.cluster import KMeans

In [22]:
X3 = df_encoded[['rating' , 'rating_count' ,'cusine','main_city','cost']].iloc[: , :].values

algorithm = (KMeans(n_clusters = 5 ,init='k-means++') )
algorithm.fit(X3)
labels3 = algorithm.labels_
centroids3 = algorithm.cluster_centers_

y_kmeans = algorithm.fit_predict(X3)
df['cluster'] = pd.DataFrame(y_kmeans)
display(df)

KeyError: "['cusine', 'main_city'] not in index"